# Skiza — CNN speech denoising (Colab walkthrough)

This notebook trains Skiza's 2-layer causal CNN on 250 noisy/clean audio pairs, evaluates it on 50 held-out pairs and lets you listen to the result.

*Runtime → Run all.* A GPU is optional: training takes under a minute on CPU.

In [ ]:
!git clone -q https://github.com/kyliemagembe/skiza.git
%cd skiza
!pip -q install -r requirements.txt

## 1. The maths in code

Each layer computes $X_\ell = \tanh\left(\sum_{k} S^k X_{\ell-1} H_{\ell k}\right)$, where $S$ is the one-sample delay.

In [ ]:
import torch
from skiza import CNN, shift, count_parameters

model = CNN(taps=[40, 40], features=[1, 5, 1])
print(model)
print('parameters:', count_parameters(model))

# Shift-equivariance check: delaying the input delays the output
x = torch.randn(1, 1, 1000) * 0.3
with torch.no_grad():
    print('Phi(Sx) == S Phi(x):', torch.allclose(model(shift(x, 25)), shift(model(x), 25), atol=1e-6))

## 2. Train and evaluate

In [ ]:
!python -m pytest -q
!python -m skiza.train

In [ ]:
import json
m = json.load(open('results/metrics.json'))
print(f"Noisy input test L1 : {m['noisy_input']['test_l1']:.5f}")
print(f"Skiza CNN test L1   : {m['skiza_mimo_cnn']['test_l1']:.5f}")
print(f"Reduction           : {m['test_l1_reduction_vs_noisy_input_pct']:.1f}%")

In [ ]:
from IPython.display import Image, display
display(Image('results/loss_curves.png'))
display(Image('results/example_test_sample.png'))

## 3. Listen

In [ ]:
from IPython.display import Audio
for name in ['noisy', 'skiza', 'clean']:
    print(name)
    display(Audio(f'results/audio/{name}.wav'))